<a href="https://colab.research.google.com/github/Abit101/IDP/blob/AbiT/01_veralto_data_sourcing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Investment Data Pipeline - Veralto
Goal: source and standardize public company metadata for VLTO - the what and why

## Section 1: Entity Metadata

This notebook begins an end-to-end investment data pipeline for sourcing,
validating, transforming, and analyzing public investment data

Veralto Corporation (NYSE: VLTO) serves as the initial case study

### Objective

Retrieve public company metadata from SEC EDGAR and transform it into
a standardized entity record that can eventually support additional
companies, REITs, and ETFs

In [67]:
#Imports
import requests #Lets Python communicate w/ websites and APIs - used to request data from SEC
import pandas as pd #Gives DataFrames (organizes data like a table or array), cleaned financial data will move through Pandas ans Spark DataFrames
from datetime import datetime, timezone #Record when pipeline retrieve the data (purpose: data lineage/provenance metadata)

#Entity Config
Define the company being processed - who and how

The pipeline initially uses Veralto Corp., butthe entity config will later be separated from the processing logic so additional companies can use the same pipeline

In [68]:
#Define Veralto
TICKER = "VLTO" #Stock-market ticker, use for market data
CIK = "1967680" #Central Index Key, SEC identifier
COMPANY_NAME = "Veralto Corp." #Readable company name

print(f"Company: {COMPANY_NAME}")
print(f"Ticker: {TICKER}")
print(f"CIK: {CIK}")

Company: Veralto Corp.
Ticker: VLTO
CIK: 1967680


##SEC EDGAR Data Source
The SEC (Securities and Exchange Comission) provides public company filing and metadata through EDGAR - where

The SEC CIK (Central Index Key) is used to identify the company when requesting its submissions data

In [69]:
#SEC Req.
HEADERS = {
    "User-Agent": "InvestmentDataPipeline abigail.qnt.tran@gmail.com"
} #When Python requests info from a website, it sends info about itself and SEC wants auto users to id themselves- "This req. comes from IDP, and who operates it" (respecting the rules of systems consume/use)

cik_padded = CIK.zfill(10) #SEC urls expect cik vals formatted as 10 digits - 1967680 -> 0001967680

url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json" #Contructs SEC endpoint dynamically, more reusability

print(url)

https://data.sec.gov/submissions/CIK0001967680.json


In [70]:
#Retrieve SEC data
response = requests.get(url, headers = HEADERS) #Python sends a request to SEC server and stores in response

response.raise_for_status() #Checks if request successfull -> stops execution if error instead of processing bad data

sec_data = response.json() #SEC response is in JSON so converts into Python data struct - nested dictionaries or lsits

print("HTTP Status: ", response.status_code)
print("Data retrieved successfully.")

HTTP Status:  200
Data retrieved successfully.


##Explore Source Data

Before transfroming the SEC response, inspect important fields to understand the structure and verify that the requested entity is Veralto - data exploration/profiling before data transformation

Need to understand raw structure before designing transformations

In [71]:
#Inspect SEC fields
sec_data.keys() #sec_data is a data_type: dictionary (key->val)

#Check source schema
print("Company: ", sec_data["name"])
print("Tickers: ", sec_data["tickers"])
print("Exchanges: ", sec_data["exchanges"])
print("SIC: ", sec_data["sic"]) #Standard Industrial Classification, used by US gov to classify companies by their primary type of business - older classification system, useful with SEC, but use case dependent
print("Industry: ", sec_data["sicDescription"])
#Might need to add other classifcation system like sector/industry if relevant


Company:  Veralto Corp
Tickers:  ['VLTO']
Exchanges:  ['NYSE']
SIC:  3825
Industry:  Instruments For Meas & Testing of  Electricity & Elec Signals


##Standardize Entity Metadata

The SEC source data is transformed into a standardize entity schema

Standardization separates the source-specific format from the format used by downstream components of the investment data pipeline - defining schema to once naming convention from different providers

In [72]:
#Data transformattion
entity_record = {
    "ticker": TICKER,
    "cik": CIK,
    "company_name": sec_data["name"],
    "exchange": sec_data["exchanges"][0]
      if sec_data["exchanges"]
      else None, #SEC gives exchanges as a list, needs single value so select first item in list and handles empty list case
    "sic": sec_data.get("sic"), #get in case field is empty or doesn't exist
    "sic_description": sec_data.get("sicDescription"),
    "source": "SEC EDGAR",
    "retrieved_at": datetime.now(timezone.utc).isoformat() #Records when data was retrieved
} #data_type: dictionary

entity_df = pd.DataFrame([entity_record]) #turns dictionary to table

entity_df

,ticker,cik,company_name,exchange,sic,sic_description,source,retrieved_at
0,VLTO,1967680,Veralto Corp,NYSE,3825,Instruments For Meas & Testing of Electricity...,SEC EDGAR,2026-08-14T20:46:08.619143+00:00


##Data Quality Validation

Before downstream use, req. entity fields are checked for missing vals - prevents incomplete entity records from silently entering later stages of the pipeline

In [73]:
required_fields = [
    "ticker",
    "cik",
    "company_name",
    "exchange",
    "source"
] #Data-quality rule, entity isn't valid unless fields exists

missing_fields = [
    field
    for field in required_fields
    if pd.isna(entity_df.loc[0, field]) #row 0 col field, None/NaN - missing val isna
    or entity_df.loc[0, field] == "" #Checks empty str
] #Check required fields and collects which is missing

if missing_fields:
  print("FAIL - Missing required fields: ", missing_fields)
else:
  print("PASS - Required entity metadata is complere.")

PASS - Required entity metadata is complere.


##Recap Section 1

1. Source - Where does the data come from?

2. Identifier - How does that system identify the company?

3. Ingestion - Ho do I retrieve the data?

4. Raw schema - What format did the source give?

5. Standardization - How do I convert that into specified format?

6. Validation - How do I determine whether the resulting data is usable?